In [1]:
# Pipeline Configuration
RUN_MODE = "test"  # "test" or "production"
TEST_SAMPLE_SIZE = 10
MAX_WORKERS = 4
CHUNK_SIZE = 500
MAX_RETRIES = 5
TIMEOUT = 30
ENABLE_CACHE = True
ENABLE_CHECKPOINT = True
SAVE_PREVIEW_IMAGES = True
PIPELINE_VERSION = "1.0.0"


# Pipeline Orchestrator\nExecute the entire pipeline in sequence and manage state.

In [2]:
import os
import json
import time
import datetime
import nbformat
from nbconvert.preprocessors import ExecutePreprocessor

state_file = "../data/metadata/pipeline_state.json"
pipeline_version = "1.0.0"

# Initialize state
if os.path.exists(state_file):
    with open(state_file, "r") as f:
        state = json.load(f)
    print(f"Resuming pipeline. Completed stages: {state['completed_notebooks']}")
else:
    state = {
        "current_notebook": "",
        "completed_notebooks": [],
        "current_POINT_ID": None,
        "completed_samples": 0,
        "failed_samples": 0,
        "execution_start_time": datetime.datetime.now().isoformat(),
        "last_update": datetime.datetime.now().isoformat(),
        "pipeline_version": pipeline_version
    }
    os.makedirs(os.path.dirname(state_file), exist_ok=True)
    with open(state_file, "w") as f:
        json.dump(state, f, indent=4)


In [3]:
notebooks_to_run = [
    ("07_lucas.ipynb", ""),
    ("01_download_data.ipynb", ""),
    ("02_sentinel2.ipynb", ""),
    ("03_sentinel1.ipynb", ""),
    ("04_dem.ipynb", ""),
    ("05_weather.ipynb", ""),
    ("06_soilgrids.ipynb", ""),
    ("01_merge_features.ipynb", "08_feature_engineering"),
    ("03_scaling.ipynb", "08_feature_engineering"),
    ("05_dataset_validation.ipynb", "08_feature_engineering"),
    ("08_feature_statistics.ipynb", "08_feature_engineering"),
    ("06_create_final_dataset.ipynb", "08_feature_engineering")
]

ep = ExecutePreprocessor(timeout=1800, kernel_name='python3')

for nb_name, subdir in notebooks_to_run:
    if nb_name in state["completed_notebooks"]:
        print(f"Skipping {nb_name} (already completed).")
        continue
        
    nb_path = os.path.join(subdir, nb_name)
    print(f"Executing {nb_path}...")
    state["current_notebook"] = nb_name
    state["last_update"] = datetime.datetime.now().isoformat()
    with open(state_file, "w") as f:
        json.dump(state, f, indent=4)
        
    try:
        with open(nb_path, encoding='utf-8') as f:
            nb_obj = nbformat.read(f, as_version=4)
        # execute
        ep.preprocess(nb_obj, {'metadata': {'path': subdir if subdir else '.'}})
        with open(nb_path, 'w', encoding='utf-8') as f:
            nbformat.write(nb_obj, f)
            
        state["completed_notebooks"].append(nb_name)
        print(f"Successfully completed {nb_name}")
    except Exception as e:
        print(f"CRITICAL FAILURE in {nb_name}: {e}")
        break
        
state["last_update"] = datetime.datetime.now().isoformat()
with open(state_file, "w") as f:
        json.dump(state, f, indent=4)
print("Pipeline run finished.")


Executing 07_lucas.ipynb...


Successfully completed 07_lucas.ipynb
Executing 01_download_data.ipynb...


Successfully completed 01_download_data.ipynb
Executing 02_sentinel2.ipynb...


Successfully completed 02_sentinel2.ipynb
Executing 03_sentinel1.ipynb...


Successfully completed 03_sentinel1.ipynb
Executing 04_dem.ipynb...


...

Successfully completed 01_merge_features.ipynb
Executing 08_feature_engineering\03_scaling.ipynb...


Successfully completed 03_scaling.ipynb
Executing 08_feature_engineering\05_dataset_validation.ipynb...


Successfully completed 05_dataset_validation.ipynb
Executing 08_feature_engineering\08_feature_statistics.ipynb...


Successfully completed 08_feature_statistics.ipynb
Executing 08_feature_engineering\06_create_final_dataset.ipynb...


Successfully completed 06_create_final_dataset.ipynb
Pipeline run finished.
